# 🤗 x 🦾: Training ACT with LeRobot Notebook

Welcome to the **LeRobot ACT training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `ACT` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `ACT` policy for 100,000 steps typically takes **about 1.5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer.

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg, and installs the package in editable mode with train and dataset features.

In [27]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install ffmpeg
!cd lerobot && pip install -e ".[train, dataset]"

fatal: destination path 'lerobot' already exists and is not an empty directory.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.6.1-0.editable-py3-none-any.whl size=14166 sha256=d18455cd26d867ef30a36d22d550bf0aaa8d6039dc1f5aef3d1a8559a42ce66b
  Stored in directory: /tmp/pip-ephem-wheel-cache-ikg4schm/wheels/09/b4/fe/75732b1d640db96ba1f856f2b7328b232a03b696a39cb59686
Successfully built lerobot
  Attempting uninstall: lerobot
    Found existing installation: lerobot 0.6.1

## Weights & Biases login (optional)
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging. This step is optional, you can skip it. If you want to use W&B remember to change `--wandb.enable` to true in the next section.

In [ ]:
# !wandb login

## HF login

To upload your trained model to the hub you need to login with your Hugging Face account.
1. Run the cell below.
2. You will be asked to generate a token in the Hugging Face settings.
3. Select all checkboxes under Repositories when creating the token.
4. Paste the generated token into the command line below.

In [1]:

# !hf auth login
# run above in shell (there is a terminal in this website)
# and copy token into HF_TOKEN secret.

## Start training ACT with LeRobot

This cell runs `lerobot-train` to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [Action Chunking with Transformers](https://huggingface.co/docs/lerobot/act), which will automatically adapt to your dataset’s setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this. Set to `False` if you do not plan on using Weights & Biases.

7. `--batch_size=8`:  
   Increase it if you memmory allows it. It defines how many datapoints are processed at once.

8. `--steps=20000`:  
   Set for how many steps you want to train your model. 20 000 steps should work fine for a simple ACT policy.

In [ ]:
version_string = "v3"

hf_user: garagelab-duesseldorf
dataset_name: pap_green_foam_in_box

output: outputs/train/pap_green_foam_in_box-v3
job_name: pap_green_foam_in_box
dataset_repo_id: garagelab-duesseldorf/pap_green_foam_in_box
policy_repo_id: garagelab-duesseldorf/pap_green_foam_in_boxpolicy-v3


In [ ]:
from google.colab import userdata
hf_user=userdata.get('HF_USER')
dataset_name=userdata.get('DATASET_NAME')
output_dir="outputs/train/" + dataset_name + "-" + version_string
job_name=dataset_name
print(f"output: {output_dir}\nhf_user: {hf_user}\ndataset_name: {dataset_name}")
print(f"job_name: {job_name}")
dataset_repo_id=hf_user + "/" + dataset_name
policy_repo_id=hf_user + "/" + dataset_name + "policy" + "-" + version_string
print(f"dataset_repo_id: {dataset_repo_id}\npolicy_repo_id: {policy_repo_id}")

In [ ]:
!lerobot-train \
  --dataset.repo_id='{dataset_repo_id}' \
  --policy.type=act \
  --output_dir='{output_dir}' \
  --job_name='{job_name}' \
  --policy.device=cuda \
  --wandb.enable=False \
  --policy.repo_id='{policy_repo_id}' \
  --num_workers=2 \
  --batch_size=32 \
  --save_checkpoint_to_hub=true \
  --save_freq=1000 \
  --steps=20000

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO 2026-08-01 15:43:28 ot_train.py:272 {'batch_size': 32,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataloader_multiprocessing_context': 'spawn',
 'dataset': {'depth_output_unit': 'mm',
             'episodes': None,
             'eval_split': 0.0,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                 

In [6]:
from huggingface_hub import HfApi

hub_api = HfApi()

# Assuming policy_repo_id is already defined from a previous cell
repo_refs = hub_api.list_repo_refs(repo_id=policy_repo_id, repo_type='model')
tag_names = [tag.name for tag in repo_refs.tags]

if 'step-5000' in tag_names:
    print(f"Der Tag 'step-5000' existiert im Repository '{policy_repo_id}'.")
else:
    print(f"Der Tag 'step-5000' existiert NICHT im Repository '{policy_repo_id}'.")

print("Verfügbare Tags:")
for tag_name in tag_names:
    print(f"- {tag_name}")

Der Tag 'step-5000' existiert NICHT im Repository 'garagelab-duesseldorf/pap_green_foam_in_boxpolicy-v3'.
Verfügbare Tags:
- 005000
- 004000
- 003000
- 002000
- 001000


In [ ]:
# Bestätigen Sie den genauen Tag des letzten Checkpoints auf dem Hugging Face Hub, bevor Sie diesen Befehl ausführen.
# Zum Beispiel, wenn der letzte Checkpoint-Tag 'step-5000' ist:
last_checkpoint_tag = "005000"

# Oder, wenn Sie den neuesten Push verwenden möchten (oft 'main' oder der Standard-Branch):
# last_checkpoint_tag = "main"

!lerobot-train \
  --dataset.repo_id='{dataset_repo_id}' \
  --policy.type=act \
  --output_dir='{output_dir}' \
  --job_name='{job_name}' \
  --policy.device=cuda \
  --wandb.enable=False \
  --policy.repo_id='{policy_repo_id}' \
  --num_workers=2 \
  --batch_size=32 \
  --save_checkpoint_to_hub=true \
  --save_freq=1000 \
  --steps=20000 \
  --policy.pretrained_path='{policy_repo_id}@{last_checkpoint_tag}'


## ATTENTION

We should do this on recording upload.


In [19]:
from huggingface_hub import HfApi

hub_api = HfApi()
hub_api.create_tag(dataset_repo_id, tag="v3.0", repo_type="dataset")

Sometimes after training, you may notice that the model underperforms and cannot solve the task properly. Sometimes this is due to poor data quality, but sometimes the model simply needs more training. To continue training from a previously trained model, use `--policy.pretrained_path=username/path_to_model` and paste the path to the model you trained previously here.